In [5]:
import requests
import pandas as pd
import time

HEADERS = {
    "User-Agent": "weather-downloader/1.0"
}

def scarica_comuni_provincia_osm(provincia):

    print(f"Ricerca provincia: {provincia}")

    # Trova la provincia
    search_url = "https://nominatim.openstreetmap.org/search"

    params = {
        "q": f"Provincia di {provincia}, Italia",
        "format": "jsonv2",
        "limit": 1
    }

    r = requests.get(
        search_url,
        params=params,
        headers=HEADERS,
        timeout=60
    )

    r.raise_for_status()

    risultati = r.json()

    if not risultati:
        raise ValueError(f"Provincia non trovata: {provincia}")

    osm_id = risultati[0]["osm_id"]

    # Overpass query
    overpass_query = f"""
    [out:json][timeout:300];

    area({3600000000 + int(osm_id)})->.searchArea;

    (
      relation["boundary"="administrative"]["admin_level"="8"](area.searchArea);
    );

    out center;
    """

    r = requests.post(
        "https://overpass-api.de/api/interpreter",
        data=overpass_query,
        headers=HEADERS,
        timeout=300
    )

    r.raise_for_status()

    data = r.json()

    records = []

    for el in data.get("elements", []):

        nome = el.get("tags", {}).get("name")

        if not nome:
            continue

        lat = el.get("center", {}).get("lat")
        lon = el.get("center", {}).get("lon")

        if lat is None or lon is None:
            continue

        records.append({
            "comune": nome,
            "lat": lat,
            "lon": lon
        })

    df = pd.DataFrame(records)

    df = (
        df
        .drop_duplicates("comune")
        .sort_values("comune")
        .reset_index(drop=True)
    )

    output_file = (
        provincia.lower()
        .replace(" ", "_")
        + "_comuni.csv"
    )

    df.to_csv(output_file, index=False)

    print(f"Salvato: {output_file}")
    print(f"Comuni trovati: {len(df)}")

    return df


# esempio
df = scarica_comuni_provincia_osm("Bergamo")

print(df.head())

Ricerca provincia: Bergamo
Salvato: bergamo_comuni.csv
Comuni trovati: 243
                   comune        lat      lon
0      Adrara San Martino  45.715882  9.96190
1        Adrara San Rocco  45.730041  9.97034
2  Albano Sant'Alessandro  45.687405  9.77294
3                  Albino  45.758951  9.81988
4                   Algua  45.817611  9.73468
